# SiC XRT 첫 분류 모델 학습

이 PC의 데이터셋을 경로로 읽습니다. 업로드하지 않아도 됩니다. 오른쪽 위 커널을 **SiC XRT GPU**로 선택한 뒤 위에서 아래로 Shift+Enter 또는 Run → Run All Cells로 실행하세요.

주석 중심 패치의 BPD/TED/TSD를 분류하는 연구용 기준 모델입니다. 정상 클래스와 전체 영상의 결함 위치 검출은 포함하지 않습니다. 제공된 ROI 라벨의 검수 상태는 결과에 남습니다.


## 1. 데이터셋 경로와 학습 설정

데이터셋 폴더 위치가 다르면 DATASET_ROOT만 바꾸세요. 처음에는 128 크기로 사용합니다. EPOCHS는 학습 반복 횟수입니다. GPU 메모리가 부족하면 BATCH_SIZE를 16 또는 8로 줄이세요.


In [ ]:
import os
from pathlib import Path

import torch

from sic_xrt_ml.training.patch_classifier import (
    dataset_info,
    evaluate_test,
    plot_history,
    train,
)

DATASET_ROOT = Path(os.environ.get("SIC_XRT_DATASET_ROOT", ""))
if not DATASET_ROOT.is_dir() or str(DATASET_ROOT) == ".":
    raise RuntimeError("SIC_XRT_DATASET_ROOT를 설정하거나 DATASET_ROOT를 실제 데이터셋 폴더로 직접 바꾸세요.")
REPO = Path(os.environ["SIC_XRT_REPO"])
OUTPUT_ROOT = REPO / "runs"
EPOCHS = 15
BATCH_SIZE = 32
PATCH_SIZE = 128
SEED = 42
assert torch.cuda.is_available(), "SiC XRT GPU 커널을 선택하세요."
print("GPU:", torch.cuda.get_device_name())
print("데이터셋:", DATASET_ROOT)


## 2. 자료 개수와 분할 확인

학습은 균형 목록을, 검증은 val 전체를 사용합니다. test는 학습 중 사용하지 않습니다.


In [ ]:
info = dataset_info(DATASET_ROOT, size=PATCH_SIZE, balanced=True)
print("클래스별 개수:", info["counts"])
print("웨이퍼 분할:", info["wafer_groups"])
print("주석 검수 상태:", info["review_status"])


## 3. 학습 시작

이 셀을 실행하면 학습합니다. epoch마다 손실과 검증 macro F1이 출력됩니다. 실행 중인 셀은 [*]로 표시됩니다. 중단하려면 상단 ■ 버튼을 누르세요. 완료한 epoch의 기록과 best 모델은 저장됩니다.


In [ ]:
result = train(DATASET_ROOT, OUTPUT_ROOT, epochs=EPOCHS, batch_size=BATCH_SIZE,
               size=PATCH_SIZE, balanced=True, seed=SEED, device="cuda")
RUN_DIR = Path(result["run_dir"])
print("완료:", result)


## 4. 결과 그래프와 저장 위치

Loss는 낮아지는지, 검증 macro F1은 높아지는지 확인합니다. 한 클래스만 맞히는 모델은 정확도가 높아도 macro F1이 낮을 수 있습니다.


In [ ]:
figure = plot_history(RUN_DIR)
print("결과 폴더:", RUN_DIR)
print("모델:", RUN_DIR / "best_model.pt")
print("설정:", RUN_DIR / "config.json")


## 5. 최종 시험 평가 — 선택 사항

학습 설정을 확정한 뒤 RUN_TEST를 True로 바꾸세요. 기본값 False이므로 전체 실행해도 시험셋은 평가하지 않습니다. 시험 결과를 반복해서 보고 학습 설정을 고르면 독립적인 평가가 아닙니다.


In [ ]:
RUN_TEST = False
if RUN_TEST:
    test_result = evaluate_test(RUN_DIR, DATASET_ROOT, batch_size=BATCH_SIZE, device="cuda")
    print(test_result)
else:
    print("시험셋은 평가하지 않았습니다. 설정 확정 후 RUN_TEST=True로 실행하세요.")
